<a href="https://colab.research.google.com/github/ekatersss/praktika06/blob/master/%D0%94%D0%BE%D0%B1%D1%80%D0%BE_%D0%BF%D0%BE%D0%B6%D0%B0%D0%BB%D0%BE%D0%B2%D0%B0%D1%82%D1%8C_%D0%B2_Colab!.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import random
import threading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

from sklearn.linear_model import LinearRegression

class VirtualSensorDriver:
    """Симулированный драйвер виртуального сенсора (character device)"""

    def __init__(self, sensor_id="AI_TEMP_001"):
        self.sensor_id = sensor_id
        self.buffer = []  # буфер данных (как в DMA)
        self.running = False
        self.thread = None
        self.interrupt_callback = None  # функция, вызываемая при "прерывании"
        self.collection = []  # список для сбора записей в реальном времени

    def open(self):
        print(f"[{self.sensor_id}] Драйвер открыт")
        self.running = True
        return True

    def close(self):
        self.running = False
        if self.thread:
            self.thread.join()
        print(f"[{self.sensor_id}] Драйвер закрыт")

    def read(self, count=10):
        """Программируемый ввод/вывод (polling)"""
        if not self.buffer:
            return []
        data = self.buffer[:count]
        self.buffer = self.buffer[count:]
        return data

    def start_interrupt_mode(self, callback=None, interval=1.0):
        """Режим с прерываниями: данные генерируются в фоне"""
        self.interrupt_callback = callback
        self.thread = threading.Thread(target=self._interrupt_thread, args=(interval,))
        self.thread.daemon = True
        self.thread.start()

    def _interrupt_thread(self, interval):
        """Симуляция аппаратного прерывания"""
        while self.running:
            # Генерируем "реальные" данные сенсора
            temp = round(20 + random.gauss(0, 3), 2)  # температура
            humidity = round(45 + random.gauss(0, 10), 1)  # влажность
            timestamp = time.time()

            packet = {"time": timestamp, "temp": temp, "humidity": humidity}
            self.buffer.append(packet)

            # Сохраняем в общую коллекцию
            self.collection.append(packet)

            # Вызываем callback (аналог ISR)
            if self.interrupt_callback:
                self.interrupt_callback(packet)

            time.sleep(interval)

    def dma_read(self, size=50):
        """Симуляция DMA: одномоментное чтение большого блока"""
        if len(self.buffer) < size:
            return []
        data = self.buffer[:size]
        self.buffer = self.buffer[size:]
        return data

#===========================Задание 1

driver = VirtualSensorDriver()
driver.open()

data_poll = driver.read(5)
print("Polling данные:", data_poll)

def on_new_data(packet):
    clear_output(wait=True)
    print(f"Получено записей: {len(driver.collection)}")
    print(f"Последние данные: {packet}")

driver.start_interrupt_mode(on_new_data, interval=0.1)
time.sleep(3)

bulk = driver.dma_read(20)
if bulk:
    df = pd.DataFrame(bulk)
    display(df.head())
else:
    print("Буфер DMA пуст или недостаточно данных")

driver.close()

#===========================Задание 2
driver = VirtualSensorDriver()
driver.open()

def on_data(packet):
    count = len(driver.collection)
    if count % 20 == 0:
        clear_output(wait=True)
        print(f"Записи собраны")

driver.start_interrupt_mode(on_data, interval=0.02)

while len(driver.collection) < 200:
    time.sleep(0.1)

driver.close()
clear_output(wait=True)

#записи найдены, преобразуем в датафрейм
df = pd.DataFrame(driver.collection)
df['time_rel'] = df['time'] - df['time'].min()

fig, ax1 = plt.subplots(figsize=(12, 6))

# температура - левая ось
color_temp = 'tab:red'
ax1.set_xlabel('Время (сек)')
ax1.set_ylabel('Температура (°C)', color=color_temp)
ax1.plot(df['time_rel'], df['temp'], color=color_temp, label='Temp', alpha=0.8)
ax1.tick_params(axis='y', labelcolor=color_temp)
ax1.grid(True, linestyle='--', alpha=0.6)

# влажность - правая ось
ax2 = ax1.twinx()
color_hum = 'tab:blue'
ax2.set_ylabel('Влажность (%)', color=color_hum)
ax2.plot(df['time_rel'], df['humidity'], color=color_hum, label='Humidity', alpha=0.8)
ax2.tick_params(axis='y', labelcolor=color_hum)

plt.title(f"Анализ данных сенсора {driver.sensor_id} (200 точек)")
fig.tight_layout()
plt.show()

#===========================Задание 3
driver = VirtualSensorDriver()
driver.open()
driver.start_interrupt_mode(interval=0.01)

while len(driver.collection) < 200:
    if len(driver.collection) % 20 == 0:
        clear_output(wait=True)
        print(f"Прогресс: {len(driver.collection)}/200")
    time.sleep(0.05)

driver.close()

# преобразование данных для scikit-learn
df = pd.DataFrame(driver.collection)
X = df[['temp']].values  # признак
y = df['humidity'].values # цель

model = LinearRegression()
model.fit(X, y)

plt.figure(figsize=(10, 5))
plt.scatter(X, y, alpha=0.5, label='Данные из драйвера', color='gray')

X_line = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
y_line = model.predict(X_line)
plt.plot(X_line, y_line, color='red', label='Линия регрессии (ML)', linewidth=3)

plt.title("ML-анализ потока данных с виртуального сенсора")
plt.xlabel("Температура (°C)")
plt.ylabel("Влажность (%)")
plt.legend()
plt.grid(True)
plt.show()

print(f"Коэффициент зависимости: {model.coef_[0]:.2f}")